In [ ]:
import requests
import pandas as pd
import time

codigo_estacao = 'A612' 
lista_dataframes = [] 

meses_inicio = pd.date_range(start='2023-01-01', end='2023-12-01', freq='MS') 
meses_fim = pd.date_range(start='2023-01-01', end='2023-12-31', freq='ME')

print("Iniciando a extração MÊS a MÊS para não sobrecarregar o INMET...\n")

for inicio, fim in zip(meses_inicio, meses_fim):
    data_inicio = inicio.strftime('%Y-%m-%d')
    data_fim = fim.strftime('%Y-%m-%d')
    
    url = f"https://apitempo.inmet.gov.br/estacao/{data_inicio}/{data_fim}/{codigo_estacao}"
    
    print(f"Buscando: {data_inicio} até {data_fim}...", end=" ")
    
    response = requests.get(url)
    
    if response.status_code == 200:
        dados_mes = response.json()
        if len(dados_mes) > 0:
            df_mes = pd.DataFrame(dados_mes)
            lista_dataframes.append(df_mes)
            print("Sucesso! [200]")
        else:
            print("Veio vazio!")
    else:
        print(f"Falhou. Status: {response.status_code}")
    
    time.sleep(1.5) 

if len(lista_dataframes) > 0:
    print("\nJuntando e limpando os dados...")
    df_completo = pd.concat(lista_dataframes, ignore_index=True)
    
    colunas_numericas = ['TEM_MAX', 'TEM_MIN', 'TEM_INS', 'CHUVA'] 
    for col in colunas_numericas:
        if col in df_completo.columns:
            df_completo[col] = df_completo[col].astype(str).str.replace(',', '.')
            df_completo[col] = pd.to_numeric(df_completo[col], errors='coerce')

    df_completo['DT_MEDICAO'] = pd.to_datetime(df_completo['DT_MEDICAO'])

    df_diario = df_completo.groupby('DT_MEDICAO').agg({
        'TEM_MAX': 'max',      
        'TEM_MIN': 'min',      
        'TEM_INS': 'mean',     
        'CHUVA': 'sum'         
    }).reset_index()

    df_diario.rename(columns={'TEM_INS': 'TEMP_MEDIA_DIA', 'CHUVA': 'CHUVA_TOTAL_DIA'}, inplace=True)
    
    print("\nExtração MENSAL concluída com sucesso! Aqui estão os primeiros dias de 2023:")
    display(df_diario.head())
else:
    print("\nInfelizmente todos os meses falharam ou retornaram vazios.")

Iniciando a extração MÊS a MÊS para não sobrecarregar o INMET...

Buscando: 2023-01-01 até 2023-01-31... Falhou. Status: 204
Buscando: 2023-02-01 até 2023-02-28... Falhou. Status: 204
Buscando: 2023-03-01 até 2023-03-31... Falhou. Status: 204
Buscando: 2023-04-01 até 2023-04-30... Falhou. Status: 204
Buscando: 2023-05-01 até 2023-05-31... Falhou. Status: 204
Buscando: 2023-06-01 até 2023-06-30... Falhou. Status: 204
Buscando: 2023-07-01 até 2023-07-31... Falhou. Status: 204
Buscando: 2023-08-01 até 2023-08-31... Falhou. Status: 204
Buscando: 2023-09-01 até 2023-09-30... Falhou. Status: 204
Buscando: 2023-10-01 até 2023-10-31... Falhou. Status: 204
Buscando: 2023-11-01 até 2023-11-30... Falhou. Status: 204
Buscando: 2023-12-01 até 2023-12-31... Falhou. Status: 204

Infelizmente todos os meses falharam ou retornaram vazios.


In [ ]:
import requests
import pandas as pd
import zipfile
import io

ano = "2023"
url_inmet = f"https://portal.inmet.gov.br/uploads/dadoshistoricos/{ano}.zip"

print(f"Baixando o pacote gigante do INMET para {ano}...")

response = requests.get(url_inmet, verify=False) 

if response.status_code == 200:
    print("\nDownload concluído! Vasculhando o ZIP na memória...")
    
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        
        arquivo_vitoria = None
        for nome_arquivo in z.namelist():
            if "A612" in nome_arquivo:
                arquivo_vitoria = nome_arquivo
                break 
        
        if arquivo_vitoria:
            print(f"Sucesso! Encontramos o arquivo: {arquivo_vitoria}")
            print("Extraindo e limpando os dados...")
            
            with z.open(arquivo_vitoria) as f:
                df_bruto = pd.read_csv(f, sep=';', skiprows=8, encoding='latin-1', decimal=',')
            
            if 'Unnamed: 19' in df_bruto.columns:
                df_bruto = df_bruto.drop(columns=['Unnamed: 19'])
            
            df_bruto.columns = [col.strip() for col in df_bruto.columns]
            
            print("\nExtração bruta finalizada! Aqui estão as primeiras linhas:")
            display(df_bruto.head())
            
        else:
            print(f"A estação A612 não estava dentro do ZIP de {ano}.")
else:
    print(f"Erro ao baixar o pacote. O portal do INMET respondeu com Status: {response.status_code}")

Baixando o pacote gigante do INMET para 2023...
(Isso pode levar de 1 a 2 minutos. Pegue um café e acompanhe o reloginho do VS Code ☕)


c:\Users\Mikaelly\Documents\projeto_clima_vitoria\venv_clima_vitoria\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'portal.inmet.gov.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(



Download concluído! Vasculhando o ZIP na memória...
Sucesso! Encontramos o arquivo: INMET_SE_ES_A612_VITORIA_01-01-2023_A_31-12-2023.CSV
Extraindo e limpando os dados...

Extração bruta finalizada! Aqui estão as primeiras linhas:


,Data,Hora UTC,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (Kj/m²),"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",TEMPERATURA DO PONTO DE ORVALHO (°C),TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C),TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIREÇÃO HORARIA (gr) (° (gr))","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)"
0,2023/01/01,0000 UTC,0.0,1017.5,1017.5,1016.8,NaN,25.0,21.6,25.5,25.0,21.6,21.3,81.0,78.0,81.0,347.0,3.3,0.8
1,2023/01/01,0100 UTC,0.0,1017.7,1017.9,1017.5,NaN,24.1,22.2,25.0,24.1,22.2,21.5,89.0,81.0,89.0,339.0,2.0,0.6
2,2023/01/01,0200 UTC,0.0,1017.2,1017.7,1017.2,NaN,24.3,21.7,24.5,24.1,22.3,21.7,89.0,85.0,86.0,343.0,2.7,0.9
3,2023/01/01,0300 UTC,0.0,1016.7,1017.2,1016.6,NaN,24.6,21.1,24.7,24.2,22.0,21.1,87.0,81.0,81.0,328.0,2.6,0.6
4,2023/01/01,0400 UTC,0.0,1016.0,1016.7,1016.0,NaN,23.5,21.9,24.6,23.5,21.9,21.1,90.0,81.0,90.0,315.0,1.5,0.5


In [ ]:
import requests
import pandas as pd
import zipfile
import io
import urllib3
import os

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

anos_para_extrair = [2021, 2022, 2023, 2024, 2025]
codigo_estacao = "A612" # Vitória - ES
lista_dataframes = []

print("Iniciando a extração definitiva. Pegue um café, isso vai levar uns minutos...\n", flush=True)

for ano in anos_para_extrair:
    url_inmet = f"https://portal.inmet.gov.br/uploads/dadoshistoricos/{ano}.zip"
    print(f"📥 Baixando e processando o pacote do ano {ano}...")
    
    response = requests.get(url_inmet, verify=False, timeout=30)
    
    if response.status_code == 200:
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            arquivo_vitoria = None
            for nome in z.namelist():
                if codigo_estacao in nome:
                    arquivo_vitoria = nome
                    break
            
            if arquivo_vitoria:
                with z.open(arquivo_vitoria) as f:
                    # Lê o CSV pulando o cabeçalho inútil
                    df_ano = pd.read_csv(f, sep=';', skiprows=8, encoding='latin-1', decimal=',')
                    
                    # Remove coluna fantasma se existir
                    if 'Unnamed: 19' in df_ano.columns:
                        df_ano = df_ano.drop(columns=['Unnamed: 19'])
                    
                    lista_dataframes.append(df_ano)
                    print(f"   Dados de {ano} extraídos com sucesso!")
            else:
                print(f"    Estação {codigo_estacao} não encontrada no ano {ano}.")
    else:
        print(f" Erro ao baixar {ano} (Status: {response.status_code})")

if lista_dataframes:
    print("\nConsolidando todos os anos e calculando as médias diárias...")
    df_completo = pd.concat(lista_dataframes, ignore_index=True)
    
    col_data = [col for col in df_completo.columns if 'Data' in col or 'DATA' in col][0]
    col_chuva = [col for col in df_completo.columns if 'PRECIPITA' in col][0]
    col_tmax = [col for col in df_completo.columns if 'MÁXIMA' in col][0]
    col_tmin = [col for col in df_completo.columns if 'MÍNIMA' in col][0]
    col_tmed = [col for col in df_completo.columns if 'BULBO SECO' in col][0]
    
    df_filtrado = df_completo[[col_data, col_chuva, col_tmax, col_tmin, col_tmed]].copy()
    df_filtrado.columns = ['DATA', 'CHUVA', 'TEMP_MAX', 'TEMP_MIN', 'TEMP_MEDIA']
    
    for col in ['CHUVA', 'TEMP_MAX', 'TEMP_MIN', 'TEMP_MEDIA']:
        df_filtrado[col] = pd.to_numeric(df_filtrado[col], errors='coerce')
        
    df_filtrado['DATA'] = pd.to_datetime(df_filtrado['DATA'], errors='coerce')
    
    df_diario = df_filtrado.groupby('DATA').agg({
        'CHUVA': 'sum',         
        'TEMP_MAX': 'max',      
        'TEMP_MIN': 'min',      
        'TEMP_MEDIA': 'mean'    
    }).reset_index()
    
    caminho_pasta = '../data/raw'
    os.makedirs(caminho_pasta, exist_ok=True)
    
    caminho_arquivo = f'{caminho_pasta}/inmet_vitoria_clima_consolidado_2021_2025.csv'
    df_diario.to_csv(caminho_arquivo, index=False)
    
    print(f"\nTudo pronto! Arquivo salvo na pasta do projeto: {caminho_arquivo}")
    display(df_diario.head())
else:
    print("\nA extração falhou. Nenhum dado foi processado.")

🚀 Iniciando a extração definitiva. Pegue um café, isso vai levar uns minutos...

📥 Baixando e processando o pacote do ano 2021...
   ✅ Dados de 2021 extraídos com sucesso!
📥 Baixando e processando o pacote do ano 2022...
   ✅ Dados de 2022 extraídos com sucesso!
📥 Baixando e processando o pacote do ano 2023...
   ✅ Dados de 2023 extraídos com sucesso!
📥 Baixando e processando o pacote do ano 2024...
   ✅ Dados de 2024 extraídos com sucesso!
📥 Baixando e processando o pacote do ano 2025...
   ✅ Dados de 2025 extraídos com sucesso!

🔄 Consolidando todos os anos e calculando as médias diárias...

🎉 Tudo pronto! Arquivo salvo na pasta do projeto: ../data/raw/inmet_vitoria_clima_consolidado_2021_2025.csv


,DATA,CHUVA,TEMP_MAX,TEMP_MIN,TEMP_MEDIA
0,2021-01-01,0.0,34.4,20.9,26.519048
1,2021-01-02,0.0,33.1,21.7,26.583333
2,2021-01-03,8.2,28.6,22.7,25.095000
3,2021-01-04,1.4,29.7,22.9,24.952174
4,2021-01-05,0.0,32.8,22.9,27.558333
